# Assignment A

By: Diwas Shrestha, Manasi Acharya and Nimesh Timalsina

## Imports

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import DepthwiseConv2D, Conv2D, SeparableConv2D

## Concept of Depthwise Convolution

<img src = "Assets/depthwise-convolution-animation-3x3-kernel.gif" alt = "Depthwise" width = "1000">

 Unlike standard convolutions that combine both spatial and channel information simultaneously, a depthwise convolution applies a single spatial filter to each input channel *independently*. Its purpose is to learn spatial patterns within each channel without mixing information across channels.

## Concept of Pointwise Convolution

<img src = "Assets/pointwise_conv.gif" alt = "Depthwise" width = "1000">

This is a standard $1 \times 1$ convolution. It is applied after the depthwise step to linearly combine the outputs of the depthwise convolution across all channels. Its purpose is to learn cross-channel patterns without looking at spatial context.

## Concept of Separable Convolution

<img src = "Assets/depthwise-separable-convolution-animation-3x3-kernel.gif" alt = "Depthwise" width = "1000">

**Depthwise Separable Convolution** factorizes a standard 3D convolution into two distinct, separate operations: a spatial filtering step (depthwise) followed by a feature combination step (pointwise). By decoupling spatial and channel filtering, it drastically reduces both the computational cost (Multiply-Accumulate operations) and the number of trainable parameters while maintaining comparable representational power.

### Implementation A: Depthwise Convolution with pointwise convolution

In [2]:
model_a = Sequential()
model_a.add(DepthwiseConv2D(kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
# Pointwise convolution
model_a.add(Conv2D(32, kernel_size=(1, 1), activation='relu'))
model_a.add(DepthwiseConv2D(kernel_size=(3,3), activation='relu'))
model_a.add(Conv2D(32, kernel_size=(1, 1), activation='relu'))

model_a.summary()

/opt/miniconda3/envs/deeplearning/lib/python3.12/site-packages/keras/src/layers/convolutional/base_depthwise_conv.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ depthwise_conv2d                │ (None, 26, 26, 1)      │            10 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 24, 24, 32)     │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 32)     │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,450 (5.66 KB)

 Trainable params: 1,450 (5.66 KB)

 Non-trainable params: 0 (0.00 B)

### Parameter Calculation

Let's trace two sequential logical blocks starting with a `28x28x1` input:

*   **1st Block** (Input: 1 channel → Output: 32 filters):
    *   *Depthwise Layer*: $(3 \times 3 \text{ kernel} \times 1 \text{ channel}) + 1 \text{ bias} = 10$ parameters.
    *   *Pointwise Layer*: $(1 \times 1 \text{ kernel} \times 1 \text{ channel} \times 32 \text{ filters}) + 32 \text{ biases} = 64$ parameters.
    *   **Block Total = 74 parameters**.

*   **2nd Block** (Input: 32 channels → Output: 32 filters):
    *   *Depthwise Layer*: $(3 \times 3 \text{ kernel} \times 32 \text{ channels}) + 32 \text{ biases} = 320$ parameters.
    *   *Pointwise Layer*: $(1 \times 1 \text{ kernel} \times 32 \text{ channels} \times 32 \text{ filters}) + 32 \text{ biases} = 1056$ parameters.
    *   **Block Total = 1376 parameters**.

**Grand Total = 74 + 1376 = 1,450 parameters**

### Implementation B: Separable Convolution

In [3]:
model_b = Sequential()
model_b.add(SeparableConv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model_b.add(SeparableConv2D(32, kernel_size=(3, 3), activation='relu'))

model_b.summary()

/opt/miniconda3/envs/deeplearning/lib/python3.12/site-packages/keras/src/layers/convolutional/base_separable_conv.py:104: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d                │ (None, 26, 26, 32)     │            73 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 24, 24, 32)     │         1,344 │
│ (SeparableConv2D)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,417 (5.54 KB)

 Trainable params: 1,417 (5.54 KB)

 Non-trainable params: 0 (0.00 B)

### Parameter Calculation

Let's trace two sequential layers starting with a `28x28x1` input:

*   **First Layer** (Input: 1 channel → Output: 32 filters):
    *   *Depthwise weights*: $(3 \times 3 \text{ kernel} \times 1 \text{ channel}) = 9$ parameters (no intermediate bias).
    *   *Pointwise weights*: $(1 \times 1 \text{ kernel} \times 1 \text{ channel} \times 32 \text{ filters}) = 32$ parameters.
    *   *Final biases*: $32$ parameters.
    *   **Layer Total = 73 parameters**.

*   **Second Layer** (Input: 32 channels → Output: 32 filters):
    *   *Depthwise weights*: $(3 \times 3 \text{ kernel} \times 32 \text{ channels}) = 288$ parameters (no intermediate bias).
    *   *Pointwise weights*: $(1 \times 1 \text{ kernel} \times 32 \text{ channels} \times 32 \text{ filters}) = 1024$ parameters.
    *   *Final biases*: $32$ parameters.
    *   **Layer Total = 1344 parameters**.

**Grand Total = 73 + 1344 = 1,417 parameters**

### Comparison of Implementation A and Implementation B

| Feature | Implementation A | Implementation B |
| :--- | :--- | :--- |
| **Architectural Design** | Two distinct layers (`DepthwiseConv2D` + `Conv2D`); applies ReLU twice (after depthwise and pointwise) | Fused `SeparableConv2D` layer; no intermediate activation (ReLU only at the end)|
| **Parameter Calculation** | **1,450 parameters** (includes intermediate bias in depthwise step). | **1,417 parameters** (omits intermediate bias in depthwise step). |
| **Computational Complexity** | $O(K^2 \cdot C_{in} + C_{in} \cdot C_{out})$ <br> Slightly higher overhead (computes intermediate ReLU and bias). | $O(K^2 \cdot C_{in} + C_{in} \cdot C_{out})$ <br> Lower overhead (no intermediate ReLU or bias). |

**Where:**
- $K$: The spatial dimension (size) of the convolutional kernel. For example, if you are using a $3 \times 3$ kernel, $K = 3$.
- $C_{in}$: The number of input channels (the depth of the input feature map) coming into the layer.
- $C_{out}$: The number of output channels (the number of filters or the depth of the output feature map) produced by the pointwise convolution layer.

## Concept of Depthwise Separable Convolution
> Both implementations represent depthwise separable convolution.

<img src = "Assets/depthwise-separable-convolution-animation-3x3-kernel.gif" alt = "Depthwise" width = "1000">

### <p style="color: #e62c4e;">Then why they produce different numbers of parameters?</p>

> In Implementation A, `DepthwiseConv2D` uses Keras's default `use_bias=True`, which adds a bias parameter (+1) after the spatial filtering step. In Implementation B, Keras's fused `SeparableConv2D` layer intentionally omits the intermediate bias during the depthwise step, only applying bias to the final pointwise output.

In [4]:
model_a = Sequential()
model_a.add(DepthwiseConv2D(kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1), use_bias=False))
# Pointwise convolution
model_a.add(Conv2D(32, kernel_size=(1, 1), activation='relu'))

model_a.add(DepthwiseConv2D(kernel_size=(3,3), activation='relu', use_bias=False))
model_a.add(Conv2D(32, kernel_size=(1, 1), activation='relu'))

model_a.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ depthwise_conv2d_2              │ (None, 26, 26, 1)      │             9 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_3              │ (None, 24, 24, 32)     │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 32)     │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,417 (5.54 KB)

 Trainable params: 1,417 (5.54 KB)

 Non-trainable params: 0 (0.00 B)

## Official Documentation & Research References

*   **Animated Depthwise and Depthwise Separable Gif:** https://animatedai.github.io/
*   **Animated Pointwise Gif:** https://medium.com/hitchhikers-guide-to-deep-learning/10-introduction-to-deep-learning-with-computer-vision-types-of-convolutions-atrous-convolutions-3cf142f77bc0
*   **DepthwiseConv2D:** https://www.tensorflow.org/api_docs/python/tf/keras/layers/DepthwiseConv2D
*   **Conv2D:** https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
*   **SeparableConv2D:** https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D